# TransformerEncoder

现在将各种组件组合起来形成TransformerEncoderBlock.

然后再将TransformerEncoderBlock堆叠N层组成TransformerEncoder.

由于组合和堆叠导致内部的注意力分数更加抽象和复杂, 注意力图已经没有必要保留, 所以去掉了 `attn_w`.

现在离散的token输入入口变为了TransformerEncoder, 需要把MultiHeadAttention里的embedding移到TransformerEncoder.

## 组件

包括MHA多头注意力, FFN前馈神经网络, LayerNorm层归一化, Resnet残差连接, Dropout.

### 多头注意力

无需多言, 我们前面一直在研究的就是多头注意力. 主要作用为解决 token 跨位置依赖, 让每个位置能动态聚合全局上下文.

### 前馈神经网络

解决位置内特征变换，没有它注意力层更像线性加权汇总，表达力不足.

### 归一化

控制特征尺度漂移, 稳定梯度与优化过程.

### 残差连接

提供短路径, 缓解深层网络梯度衰减与信息退化.

### Dropout

抑制共适应, 降低过拟合风险.

> 本文以结构机理解释为主, 不做严格消融; 结论属于理论与经验共识, 不是本教程内的实证结论.



In [ ]:
import torch
from torch import nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1): # 由于现在离散的token输入入口变为了TransformerEncoderBlock, 需要把MultiHeadAttention里的embedding移到block
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.dropout = dropout

        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.out = nn.Linear(d_model, d_model) # 注意这里的输出维度改为了d_model, 因为多头注意力的输出维度是d_model, 而不是output_dim

    def _rotate_half(self, x):
        # x: (batch, num_heads, seq_len, d_k)
        x1 = x[..., ::2]
        x2 = x[..., 1::2]
        return torch.stack((-x2, x1), dim=-1).flatten(-2)

    def _rope(self, q, k, base=10000):
        # q, k: (batch, num_heads, seq_len, d_k)
        seq_len = q.size(-2)
        freqs = torch.arange(0, self.d_k, 2, device=q.device) / self.d_k
        freqs = base ** -freqs
        angles = torch.arange(seq_len, device=q.device)[:, None] * freqs[None, :]
        sin = angles.sin().repeat_interleave(2, dim=-1)[None, None]
        cos = angles.cos().repeat_interleave(2, dim=-1)[None, None]
        return q * cos + self._rotate_half(q) * sin, \
              k * cos + self._rotate_half(k) * sin

    def forward(self, x):
        """
        x: (batch, seq_len, d_model)
        """
        batch_size = x.size(0)
        Q = self.w_q(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.w_k(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.w_v(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        Q, K = self._rope(Q, K)
        score = Q @ K.transpose(-2, -1) / self.d_k ** 0.5
        attn_w = torch.softmax(score, dim=-1)

        output = attn_w @ V
        output = output.transpose(1, 2).reshape(batch_size, -1, self.d_model)
        output = self.out(output)
        return output


class TransformerEncoderBlock(nn.Module): # 新增block
    def __init__(self, ffn_dim, d_model, num_heads, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, num_heads, dropout=dropout)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.ReLU(),
            nn.Dropout(dropout), # 新增Dropout层
            nn.Linear(ffn_dim, d_model),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x_norm = self.norm1(x)
        x = x + self.attn(x_norm)
        x_norm = self.norm2(x)
        x = x + self.ffn(x_norm)
        return x


class TransformerEncoder(nn.Module): # 新增encoder
    def __init__(self, vocab_size, output_dim, ffn_dim, d_model, num_heads, num_layers, dropout=0.1): # 将embedding放在最外层，作为离散token的输入入口
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([ # 通过ModuleList堆叠多个block
            TransformerEncoderBlock(ffn_dim, d_model, num_heads, dropout=dropout)
            for _ in range(num_layers)
        ])
        self.out = nn.Linear(d_model, output_dim)

    def forward(self, x):
        x = self.embedding(x)          # (batch, seq_len, d_model)
        for layer in self.layers:
            x = layer(x)
        return self.out(x)

TransformerEncoderBlock里的forward有两种写法, 一种是上面的Pre-Norm写法.

一种是下面的Transformer论文的经典写法Post-Norm, 先做注意力/FFN, 并残差连接, 再norm.

```python
def forward(self, x):
        x = x + self.attn(x) # (batch, seq_len, output_dim) 先注意力并残差
        x = self.norm1(x) # 再层归一化
        x = x + self.ffn(x) # (batch, seq_len, d_model) 先FFN并残差
        x = self.norm2(x) # 再层归一化
        return x
```
一般来说, Pre-Norm写法对于深层网络训练更稳定, 且是现在大模型的主流写法, 所以我们用这种.

## 实验1：序列排序


In [ ]:
def get_sort_batch(batch_size=64, seq_len=6, vocab_size=10):
    """
    排序任务数据集
    输入: 乱序的整数序列      e.g. [3, 1, 4, 1, 5, 2]
    标签: 升序排列后的序列    e.g. [1, 1, 2, 3, 4, 5]
    """
    x = torch.randint(0, vocab_size, (batch_size, seq_len), dtype=torch.long)
    y = x.sort(dim=-1).values
    return x, y

# 验证一下
x, y = get_sort_batch(4, 6, 10)
for i in range(4):
    print(f"input: {x[i].tolist()}  →  sorted: {y[i].tolist()}")

epochs = 200

model = TransformerEncoder(
    vocab_size=10,
    output_dim=10,
    ffn_dim=64,
    d_model=64,
    num_heads=4,
    num_layers=2,
    dropout=0.1
)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

model.train()
for i in range(epochs):
    train_x, train_y = get_sort_batch(1024, 10, 10)
    optimizer.zero_grad()
    pred = model(train_x)                      # (batch, seq_len, vocab_size)
    loss = criterion(pred.transpose(1, 2), train_y)
    loss.backward()
    optimizer.step()
    if (i + 1) % 10 == 0:
        print(f"Epoch {i + 1}, Loss: {loss.item():.4f}")

test_x, test_y = get_sort_batch(4, 6, 10)
model.eval()
with torch.no_grad():
    pred = model(test_x)
    pred_tokens = pred.argmax(dim=-1)

for i in range(test_x.size(0)):
    print(f"input : {test_x[i].tolist()}")
    print(f"pred  : {pred_tokens[i].tolist()}")
    print(f"target: {test_y[i].tolist()}")
    print("-" * 30)

input: [7, 3, 9, 9, 1, 6]  →  sorted: [1, 3, 6, 7, 9, 9]
input: [0, 1, 6, 6, 5, 5]  →  sorted: [0, 1, 5, 5, 6, 6]
input: [6, 3, 8, 0, 1, 8]  →  sorted: [0, 1, 3, 6, 8, 8]
input: [5, 3, 2, 2, 3, 5]  →  sorted: [2, 2, 3, 3, 5, 5]
Epoch 100, Loss: 0.2971
Epoch 200, Loss: 0.0429
input : [3, 4, 0, 3, 8, 2]
pred  : [0, 2, 3, 3, 3, 4]
target: [0, 2, 3, 3, 4, 8]
------------------------------
input : [4, 1, 5, 6, 5, 5]
pred  : [4, 4, 5, 5, 5, 6]
target: [1, 4, 5, 5, 5, 6]
------------------------------
input : [6, 6, 9, 2, 3, 2]
pred  : [2, 2, 3, 6, 6, 6]
target: [2, 2, 3, 6, 6, 9]
------------------------------
input : [7, 2, 9, 0, 9, 7]
pred  : [0, 2, 7, 7, 9, 9]
target: [0, 2, 7, 7, 9, 9]
------------------------------


In [4]:
test_x, test_y = get_sort_batch(4, 6, 10)
model.eval()
with torch.no_grad():
    pred = model(test_x)                   # 触发 forward，缓存 attn_w
    pred_tokens = pred[0].argmax(dim=-1)   # (seq_len,)

print("input :", test_x[0].tolist())
print("pred  :", pred_tokens.tolist())
print("target:", test_x[0].sort().values.tolist())


input : [8, 5, 1, 6, 9, 6]
pred  : [5, 6, 6, 6, 8, 9]
target: [1, 5, 6, 6, 8, 9]
